# Exercises XP: LoRA Implementation Lab
Replace each `TODO` before running the next section.

## What you'll learn

- The fundamentals of LoRA (Low-Rank Adaptation) and why it helps churn out efficient fine-tunes.
- How to implement LoRA matrices `A` and `B`, plus how to wrap existing `nn.Linear` layers.
- Differences between standard linear layers, LoRA-enhanced layers, and merged-weight alternatives.
- How to freeze base parameters so that only the LoRA adapters receive updates.

## What you will create

- A reusable `LoRALayer` module and two linear wrappers (`LinearWithLoRA`, `LinearWithLoRAMerged`).
- A 3-layer MLP that can be swapped between standard and LoRA-enhanced variants.
- A minimal MNIST training loop plus accuracy helpers to compare frozen vs. fully-trainable adapters.
- A workflow to freeze baseline weights and fine-tune only the LoRA layers.

> **Learning point**  
> Keep the student and teacher notebooks open side by side. Follow the numbered exercises, run setup only once, and watch tensor shapes as you add LoRA adapters.

# Part 0: Environment Setup

Install the CPU-friendly PyTorch stack plus torchvision for MNIST. Reuse caches across reruns to save time.

In [1]:
%pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [2]:
import copy
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

BASE_SEED = 123
torch.manual_seed(BASE_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


# Exercise 1: Implement `LoRALayer`

Create the low-rank matrices `A` and `B`, scale them with `alpha`, and test the module on a toy tensor.

In [4]:
import torch
import torch.nn as nn

class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        # Low-rank decomposition: A projects down, B projects up
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha
        self.scaling = alpha / rank  # common scaling trick in LoRA

    def forward(self, x):
        # (batch, in_dim) @ (in_dim, rank) -> (batch, rank)
        # (batch, rank) @ (rank, out_dim) -> (batch, out_dim)
        update = (x @ self.A) @ self.B
        return self.scaling * update

# Hyperparameters for sandbox test
random_seed = 123
in_dim = 4
out_dim = 3
rank = 2
alpha = 4

torch.manual_seed(random_seed)
layer = LoRALayer(in_dim, out_dim, rank, alpha)

# Toy input tensor
x = torch.randn(5, in_dim)  # batch size 5, input dimension 4

print("Input:\n", x)
print("\nLayer:\n", layer)
print("\nOriginal output:\n", layer(x))

Input:
 tensor([[-0.0770, -1.0205, -0.1690,  0.9178],
        [-0.3885, -0.9343, -0.4991, -1.0867],
        [ 0.9624,  0.2492, -0.4845, -2.0929],
        [ 0.0983, -0.0935,  0.2662, -0.5850],
        [-0.3430, -0.6821, -0.9887, -1.7018]])

Layer:
 LoRALayer()

Original output:
 tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]], grad_fn=<MulBackward0>)


# Exercise 2: Wrap `nn.Linear` with LoRA

Combine a frozen linear projection plus a trainable `LoRALayer`. Confirm the adapter outputs add on top of the base logits.

In [5]:
class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)


base_linear = nn.Linear(in_dim, out_dim)
layer_lora_1 = LinearWithLoRA(base_linear, rank, alpha)  # wrap `base_linear` with rank/alpha values
print("LinearWithLoRA output:", layer_lora_1(x))

LinearWithLoRA output: tensor([[-0.8209, -0.7684, -0.7734],
        [ 0.1118,  0.1666,  0.0684],
        [ 0.9664,  0.6621,  0.9957],
        [ 0.0491,  0.1947, -0.0763],
        [ 0.4834,  0.4948,  0.6145]], grad_fn=<AddBackward0>)


# Exercise 3: Swap a simple network layer with LoRA

Start from a single-layer perceptron, then replace its linear block with `LinearWithLoRA`. The outputs should match before training because the LoRA adapters start at zero.

In [7]:
class SingleLayerNet(nn.Module):
    def __init__(self, num_features, num_classes):
        super().__init__()
        self.layer = nn.Linear(num_features, num_classes)

    def forward(self, x):
        return self.layer(x)

single_net = SingleLayerNet(num_features=4, num_classes=3)
sample_input = torch.randn(5, 4)

with torch.no_grad():
    baseline_output = single_net(sample_input)

single_net.layer = LinearWithLoRA(single_net.layer, rank=2, alpha=4)

with torch.no_grad():
    lora_output = single_net(sample_input)

print("Outputs match before training?", torch.allclose(baseline_output, lora_output))

Outputs match before training? True


# Exercise 4: Merged-weight LoRA layer

Fuse the LoRA matrices with the frozen weights to create a drop-in linear layer that behaves exactly like `LinearWithLoRA`.

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        # Produit des matrices LoRA
        lora = self.lora.A @ self.lora.B
        # Fusion avec les poids gelés
        combined_weight = self.linear.weight + self.lora.scaling * lora.T
        # Utilisation comme une couche Linear classique
        return F.linear(x, combined_weight, self.linear.bias)


# Exemple d’utilisation
layer_lora_2 = LinearWithLoRAMerged(base_linear, rank=2, alpha=4)
print("Merged LoRA output:", layer_lora_2(x))

Merged LoRA output: tensor([[-0.8209, -0.7684, -0.7734],
        [ 0.1118,  0.1666,  0.0684],
        [ 0.9664,  0.6621,  0.9957],
        [ 0.0491,  0.1947, -0.0763],
        [ 0.4834,  0.4948,  0.6145]], grad_fn=<AddmmBackward0>)


# Exercise 5: Build an MLP and prepare MNIST

Stack three linear layers with ReLU activations, then set up the MNIST loaders plus optimizer/state for pretraining.

In [9]:
class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(num_features, num_hidden_1),
            nn.ReLU(),
            nn.Linear(num_hidden_1, num_hidden_2),
            nn.ReLU(),
            nn.Linear(num_hidden_2, num_classes),
        )

    def forward(self, x):
        x = self.layers(x)
        return x

In [10]:
# Architecture
num_features = 28 * 28
num_hidden_1 = 256
num_hidden_2 = 128
num_classes = 10

# Settings
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
learning_rate = 1e-3
num_epochs = 10

model = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes,
)

model.to(DEVICE)
optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=learning_rate)

print(DEVICE)
print(model)
print(optimizer_pretrained)

cpu
MultilayerPerceptron(
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


## Loading dataset

In [11]:
BATCH_SIZE = 64

train_dataset = datasets.MNIST(
    root='data',
    train=True,
    transform=transforms.ToTensor(),
    download=True
)

test_dataset = datasets.MNIST(
    root='data',
    train=False,
    transform=transforms.ToTensor(),
    download=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

for images, labels in train_loader:
    print('Image batch dimensions:', images.shape)
    print('Image label dimensions:', labels.shape)
    break

100%|██████████| 9.91M/9.91M [00:00<00:00, 64.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.71MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.4MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.50MB/s]

Image batch dimensions: torch.Size([64, 1, 28, 28])
Image label dimensions: torch.Size([64])


## Define evaluation

In [24]:
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct_pred, num_examples = 0, 0
    with torch.no_grad():
        for features, targets in data_loader:
            features = features.view(-1, model.layers[0].linear.in_features).to(device)
            targets = targets.to(device)
            logits = model(features)
            _, predicted_labels = torch.max(logits, 1)
            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum().item()
    return correct_pred / num_examples

## Training

In [13]:
def train(num_epochs, model, optimizer, train_loader, device):
    start_time = time.time()
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, targets) in enumerate(train_loader):
            features = features.view(-1, model.layers[0].in_features).to(device)
            targets = targets.to(device)

            logits = model(features)
            loss = criterion(logits, targets)
            optimizer.zero_grad()

            loss.backward()
            optimizer.step()

            if not batch_idx % 400:
                print('Epoch: %03d/%03d|Batch %03d/%03d| Loss: %.4f' %
                      (epoch+1, num_epochs, batch_idx, len(train_loader), loss))

        with torch.set_grad_enabled(False):
            print('Epoch: %03d/%03d training accuracy: %.2f%%' %
                  (epoch+1, num_epochs, compute_accuracy(model, train_loader, device)*100))

        print('Time elapsed: %.2f min' % ((time.time() - start_time)/60))
    print('Total Training Time: %.2f min' % ((time.time() - start_time)/60))

In [15]:
# Définir la fonction de perte
criterion = nn.CrossEntropyLoss()

# Lancer l'entraînement
train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)

# Évaluer sur le jeu de test
print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE)*100:.2f}%')

/tmp/ipykernel_218/4180489156.py:17: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print('Epoch: %03d/%03d|Batch %03d/%03d| Loss: %.4f' %


Epoch: 001/010|Batch 000/938| Loss: 2.3134
Epoch: 001/010|Batch 400/938| Loss: 0.1695
Epoch: 001/010|Batch 800/938| Loss: 0.2633
Epoch: 001/010 training accuracy: 96.52%
Time elapsed: 0.32 min
Epoch: 002/010|Batch 000/938| Loss: 0.0850
Epoch: 002/010|Batch 400/938| Loss: 0.1760
Epoch: 002/010|Batch 800/938| Loss: 0.0724
Epoch: 002/010 training accuracy: 98.13%
Time elapsed: 0.66 min
Epoch: 003/010|Batch 000/938| Loss: 0.0378
Epoch: 003/010|Batch 400/938| Loss: 0.0328
Epoch: 003/010|Batch 800/938| Loss: 0.0824
Epoch: 003/010 training accuracy: 98.53%
Time elapsed: 1.00 min
Epoch: 004/010|Batch 000/938| Loss: 0.0291
Epoch: 004/010|Batch 400/938| Loss: 0.0395
Epoch: 004/010|Batch 800/938| Loss: 0.0309
Epoch: 004/010 training accuracy: 99.07%
Time elapsed: 1.35 min
Epoch: 005/010|Batch 000/938| Loss: 0.0123
Epoch: 005/010|Batch 400/938| Loss: 0.0541
Epoch: 005/010|Batch 800/938| Loss: 0.0833
Epoch: 005/010 training accuracy: 99.15%
Time elapsed: 1.67 min
Epoch: 006/010|Batch 000/938| Loss:

# Replacing Linear with LoRA Layers

In [20]:
import copy

model_lora = copy.deepcopy(model)

model_lora.layers[0] = LinearWithLoRAMerged(model_lora.layers[0], rank=4, alpha=8)
model_lora.layers[2] = LinearWithLoRAMerged(model_lora.layers[2], rank=4, alpha=8)
model_lora.layers[4] = LinearWithLoRAMerged(model_lora.layers[4], rank=4, alpha=8)

model_lora.to(DEVICE)
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)

print(model_lora)



MultilayerPerceptron(
  (layers): Sequential(
    (0): LinearWithLoRAMerged(
      (linear): Linear(in_features=784, out_features=256, bias=True)
      (lora): LoRALayer()
    )
    (1): ReLU()
    (2): LinearWithLoRAMerged(
      (linear): Linear(in_features=256, out_features=128, bias=True)
      (lora): LoRALayer()
    )
    (3): ReLU()
    (4): LinearWithLoRAMerged(
      (linear): Linear(in_features=128, out_features=10, bias=True)
      (lora): LoRALayer()
    )
  )
)


## Freezing the Original Linear Layers

In [21]:
def freeze_linear_layers(model):
    for child in model.children():
        if isinstance(child, nn.Linear):
            for param in child.parameters():
                param.requires_grad = False
        elif isinstance(child, LinearWithLoRAMerged):
            for param in child.linear.parameters():
                param.requires_grad = False
        else:
            freeze_linear_layers(child)

freeze_linear_layers(model_lora)

for name, param in model_lora.named_parameters():
    print(f'{name}: {param.requires_grad}')

layers.0.linear.weight: False
layers.0.linear.bias: False
layers.0.lora.A: True
layers.0.lora.B: True
layers.2.linear.weight: False
layers.2.linear.bias: False
layers.2.lora.A: True
layers.2.lora.B: True
layers.4.linear.weight: False
layers.4.linear.bias: False
layers.4.lora.A: True
layers.4.lora.B: True
